# EDA de logs NDN · Iteración 1 del simulador

**Objetivo:** preparar los datos para crear embeddings de *interests* que combinen dos señales:

1. **Nombre**: interests con nombres parecidos deben quedar cerca.
2. **Contexto**: interests que un mismo consumidor pide juntos, y que se repiten juntos en muchos consumidores, deben quedar cerca.

**Decisiones de diseño de la señal de contexto:**

- El **consumidor** es la cara (`node`, `face`): los 10 nodos `barrio-*` mezclan a 444 reproductores, cada uno con su cara.
- La distancia entre A y B se mide sobre todo en **pasos** dentro de la secuencia; el tiempo, como mucho, como ajuste suave que no decide. No hay un corte duro de sesiones.
- La relación es **principalmente simétrica**; el orden (A→B frente a B→A) es un matiz secundario.
- Pesa más un par repetido **en muchos consumidores** que uno repetido muchas veces por uno solo.
- La frecuencia bruta se corrige por **popularidad** (p. ej. PMI).

## 1. Carga

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

ruta = "paquetes.csv"

- `time` son segundos de simulación con precisión de milisegundos: se deja numérico, sin `parse_dates`.
- **No se aplica `drop_duplicates`**: elimina filas legítimas y descuadra los conteos posteriores.

In [3]:
df = pd.read_csv(ruta, engine="pyarrow")
print(df.shape)
df.head()

(421529, 11)


,time,node,dir,tipo,src,dst,nombre,bytes,motivo,face,nonce
0,0.0,barrio-bio-1,NaN,CsMiss,NaN,NaN,/localhost/nfd/faces/events/seq=13,0,NaN,-1,bf260cd4
1,0.0,barrio-svq-2,NaN,CsMiss,NaN,NaN,/localhost/nfd/faces/events/seq=18,0,NaN,-1,803d44ab
2,0.0,barrio-mad-1,NaN,CsMiss,NaN,NaN,/localhost/nfd/faces/events/seq=76,0,NaN,-1,ecc556fd
3,0.0,yt-canales-a,NaN,CsMiss,NaN,NaN,/localhost/nfd/faces/events/seq=158,0,NaN,-1,6cbdea64
4,0.0,barrio-vlc-1,NaN,CsMiss,NaN,NaN,/localhost/nfd/faces/events/seq=37,0,NaN,-1,cd4af6ca


## 2. Vista general del log

Tipos de evento:

- `CsMiss` / `CsHit`: consultas a la caché del nodo (sin dirección, `face = -1`).
- `Interest` / `Data` con `dir` = `in` / `out`.
- `Timeout`: eliminación de una entrada de la PIT. En el contenido hay 1.536 filas, tantas como `Nack out`, y todas son de nombres con Nack: no hay indicios de caducidades reales.
- `Nack`: siempre con `motivo = NoRoute`.

In [4]:
pd.crosstab(df["tipo"], df["dir"].fillna("—"))

dir,in,out,—
tipo,,,
CsHit,0,0,16360
CsMiss,0,0,125911
Data,44578,60938,0
Interest,62474,45602,0
Nack,1024,1536,0
Timeout,0,0,63106


In [5]:
df["prefijo1"] = df["nombre"].str.split("/").str[1]
df["prefijo1"].value_counts(dropna=False)

prefijo1
vod          151131
localhost    141367
yt           102310
rtve          26721
Name: count, dtype: int64

## 3. Filtrar tráfico interno

Los nombres `/localhost/...` son tráfico de gestión de NFD, no contenido. Se eliminan.

Los `face = -1` que quedan **no** son ruido: son los `CsMiss`, `CsHit` y `Timeout` de cada nodo y forman parte del recorrido de las peticiones.

In [6]:
dc = df[df["prefijo1"] != "localhost"].copy()
print(len(dc), "filas de contenido")

280162 filas de contenido


## 4. Peticiones de los consumidores

Una petición de usuario es un `Interest` con `dir = in` y `src = app`: la aplicación entrega el Interest a su nodo.

- Solo los 10 nodos `barrio-*` originan peticiones.
- Cada `nonce` identifica un Interest concreto en todos sus saltos.

In [7]:
req = dc[(dc["tipo"] == "Interest") & (dc["dir"] == "in") & (dc["src"] == "app")].copy()

print(len(req), "peticiones")
print(req["nonce"].nunique(), "nonces distintos")
req["node"].value_counts()

20643 peticiones
20643 nonces distintos


node
barrio-mad-1    3548
barrio-mad-2    2948
barrio-bcn-2    2615
barrio-bcn-1    2315
barrio-vlc-2    1889
barrio-bio-2    1780
barrio-svq-1    1648
barrio-svq-2    1406
barrio-vlc-1    1312
barrio-bio-1    1182
Name: count, dtype: int64

## 5. ¿Se sirvió cada petición?

Los paquetes Data **no llevan nonce**, así que se emparejan por nodo, nombre y tiempo: el primer `Data out` con `dst = app` en el mismo nodo, por la misma cara y con el mismo nombre, como mucho 4 s después de la petición.

In [8]:
entregas = (dc[(dc["tipo"] == "Data") & (dc["dir"] == "out") & (dc["dst"] == "app")]
              [["node", "face", "nombre", "time"]]
              .rename(columns={"time": "t_data"})
              .sort_values("t_data"))

req = pd.merge_asof(req.sort_values("time"), entregas,
                    left_on="time", right_on="t_data",
                    by=["node", "face", "nombre"], direction="forward", tolerance=4.0)
req["ok"] = req["t_data"].notna()
req["ok"].value_counts(normalize=True)

ok
True     0.975197
False    0.024803
Name: proportion, dtype: float64

## 6. Secuencias y colapso de repeticiones

**La unidad de secuencia es la cara (`node`, `face`), no el nodo**: agrupar por nodo intercalaría usuarios distintos y crearía co-ocurrencias falsas. Hay una secuencia por consumidor, sin cortes de sesión: qué pares cuentan lo decidirá K, en pasos.

Repeticiones consecutivas del mismo nombre:

- Tras una petición fallida, el reproductor reintenta a los 5 s tras un **Nack NoRoute** (valor fijo del simulador, `video-player.cpp:115`).
- Tras una petición servida, las de menos de 0,3 s son todas **miniaturas**.

Una racha `A A A B` no aporta más contexto que `A B`, así que se colapsa. `A B A` no se colapsa. Se guarda `n_peticiones` por si la intensidad sirve después como señal; el colapso cambia las frecuencias respecto al log original, lo que afecta al muestreo negativo.

In [9]:
clave = ["node", "face"]

req = req.sort_values(clave + ["time"]).reset_index(drop=True)
print(req.groupby("node")["face"].nunique().describe())

count    10.000000
mean     44.400000
std      13.745302
min      25.000000
25%      35.500000
50%      37.500000
75%      57.250000
max      66.000000
Name: face, dtype: float64


In [ ]:
# Comprobación antes de colapsar
repe = req["nombre"] == req.groupby(clave)["nombre"].shift()
print(repe.sum(), "repeticiones consecutivas por cara")   # esperado: 743

In [ ]:
# Colapso dentro de cada cara
req["bloque"] = (~repe).cumsum()

seq = (req.groupby("bloque")
          .agg(node=("node", "first"),
               face=("face", "first"),
               nombre=("nombre", "first"),
               time=("time", "first"),
               t_fin=("time", "last"),
               n_peticiones=("nombre", "size"),
               ok=("ok", "any"))
          .reset_index(drop=True))
seq["paso"] = seq.groupby(clave).cumcount()

print(len(req) - len(seq), "repeticiones colapsadas")   # esperado: 743
print(len(seq), "pasos en las secuencias")              # esperado: 19900
print(seq.groupby(clave).ngroups, "consumidores")       # esperado: 444

### Hueco sin el tiempo de visionado

El reproductor pide un solo Interest y luego "ve" el vídeo sin generar tráfico, así que el hueco bruto incluye el visionado. Con segmentos reales, el router vería Interests durante todo ese tiempo, de modo que se descuenta la duración del vídeo anterior.

- La duración es aproximada: la mediana de cada nombre en `qoe.csv`. Es información de la simulación que el router no tiene.
- `hueco` solo sirve para el ajuste suave por tiempo; no corta ni decide nada.


In [ ]:
qoe = pd.read_csv("qoe.csv")
duracion = (qoe[(qoe["tipo"] != "miniatura") & (qoe["resultado"] == "completa")]
               .groupby("base")["duracion_s"].median())

seq["visionado"] = seq["nombre"].map(duracion).fillna(0)
fin_anterior = (seq["t_fin"] + seq["visionado"]).groupby([seq["node"], seq["face"]]).shift()
seq["hueco"] = (seq["time"] - fin_anterior).clip(lower=0)

seq["hueco"].describe(percentiles=[.5, .9, .99])


Las peticiones con `ok = False` se mantienen porque son lo que ve el router, pero **no son lo que el usuario quería**: en las sesiones con error, el nombre pedido nunca coincide con la intención. Tampoco pueden ser destino de una redirección.

Guardamos en Parquet para no repetir la limpieza en cada reinicio del kernel.

In [ ]:
seq.to_parquet("secuencias_iter1.parquet")
# seq = pd.read_parquet("secuencias_iter1.parquet")

## 7. Efecto de las miniaturas

Una portada carga decenas de miniaturas casi a la vez, y `humanos.py` las elige por popularidad, afinidad y género, o por temática en `/yt`. Se mide qué peso tienen en los pares, cuánto alejan a los vídeos entre sí y qué señal aportan.

In [ ]:
seq["mini"] = seq["nombre"].str.endswith("/miniatura")
secuencias = [g for _, g in seq.sort_values(clave + ["paso"]).groupby(clave)]
print(f"{seq['mini'].mean():.1%} de los pasos son miniaturas")

# Composición de los pares según K
filas = []
for K in (1, 2, 5, 10, 20):
    mm = mv = vv = 0
    for g in secuencias:
        m = g["mini"].to_numpy()
        for d in range(1, K + 1):
            a, b = m[:-d], m[d:]
            mm += (a & b).sum()
            vv += (~a & ~b).sum()
            mv += (a != b).sum()
    total = mm + mv + vv
    filas.append({"K": K, "pares": total, "% MM": 100 * mm / total, "% MV": 100 * mv / total, "% VV": 100 * vv / total})
pd.DataFrame(filas).round(1)

Pasos entre un vídeo y el siguiente vídeo del mismo consumidor, con las miniaturas en medio:

In [ ]:
dist = np.concatenate([np.diff(np.flatnonzero(~g["mini"].to_numpy())) for g in secuencias])
pd.Series({f"K={K}": (dist <= K).mean() * 100 for K in (1, 2, 5, 10, 20, 50)}, name="% de vídeos consecutivos alcanzados").round(1)

¿Qué señal aportan? Se compara cuántos pares comparten tema con lo que daría el azar dentro de la misma plataforma. El tema viene del catálogo: solo se usa para medir, el router no lo conoce. Los vídeos de `/yt/shorts` se excluyen porque contarían como un único título.

In [ ]:
import sys
sys.path.insert(0, "../../video")
import catalogo as cat

temas = {t.prefijo(): set(t.generos) or {t.tipo} for t in cat.TITULOS}
temas.update({f"/yt/{c.slug}": {c.tematica} for c in cat.canales()})

def titulo(n):
    p = n.split("/")
    return "/".join(p[:3] if p[1] == "yt" else p[:4])

def mismo_tema(a, b):
    return bool(temas.get(titulo(a), set()) & temas.get(titulo(b), set()))

rng = np.random.default_rng(1)
filas = []
for mini in (True, False):
    pares = []
    for g in secuencias:
        if mini:  # miniaturas adyacentes, de la misma portada o lista
            n, m = g["nombre"].tolist(), g["mini"].tolist()
            pares += [(a, b) for a, b, x, y in zip(n, n[1:], m, m[1:]) if x and y]
        else:     # vídeos consecutivos, ignorando las miniaturas
            n = g.loc[~g["mini"], "nombre"].tolist()
            pares += list(zip(n, n[1:]))
    pares = [(a, b) for a, b in pares if "/yt/shorts/" not in a and "/yt/shorts/" not in b]
    for p in ("vod", "rtve", "yt"):
        dentro = [(a, b) for a, b in pares if a.split("/")[1] == p == b.split("/")[1]]
        grupo = seq[(seq["mini"] == mini) & seq["nombre"].str.startswith(f"/{p}/")
                    & ~seq["nombre"].str.contains("/yt/shorts/")]["nombre"].to_numpy()
        azar = zip(rng.choice(grupo, 8000), rng.choice(grupo, 8000))
        filas.append({"pares": "miniaturas" if mini else "vídeos", "plataforma": p, "n": len(dentro),
                      "% mismo tema": 100 * np.mean([mismo_tema(a, b) for a, b in dentro]),
                      "% azar": 100 * np.mean([mismo_tema(a, b) for a, b in azar]),
                      "% mismo título": 100 * np.mean([titulo(a) == titulo(b) for a, b in dentro])})
pd.DataFrame(filas).round(1)

¿Lleva la miniatura al vídeo? Y los nombres que fallan, que son los que habrá que redirigir, ¿qué tienen alrededor?

In [ ]:
videos = con_titulo = 0
for g in secuencias:
    n, m = g["nombre"].tolist(), g["mini"].tolist()
    for i in range(len(n)):
        if not m[i]:
            videos += 1
            con_titulo += titulo(n[i]) in {titulo(x) for x, y in zip(n[max(0, i - 30):i], m[max(0, i - 30):i]) if y}
print(f"Vídeos con una miniatura de su título en los 30 pasos anteriores: {con_titulo} de {videos} ({con_titulo / videos:.1%})")

fallos = seq[~seq["ok"]]
print(f"Fallos en miniaturas: {fallos['mini'].sum()} de {len(fallos)}")
n_videos = (~seq["mini"]).groupby([seq["node"], seq["face"]]).sum()
print("Fallos sin ningún otro vídeo en su secuencia:", sum(n_videos[(r.node, r.face)] == 1 for r in fallos.itertuples()))
for K in (1, 5, 10):
    con_video = 0
    for g in secuencias:
        n = g["mini"].to_numpy()
        for i in np.flatnonzero(~g["ok"].to_numpy()):
            vecinos = np.r_[n[max(0, i - K):i], n[i + 1:i + K + 1]]
            con_video += (~vecinos).any()
    print(f"Fallos con un vídeo a K={K} o menos pasos: {con_video} de {len(fallos)}")

**Resultados:**

- **Pares:** el 89,3 % de los pasos son miniaturas. Con cualquier K, unos 86 de cada 100 pares son miniatura-miniatura y entre 2 y 5 son vídeo-vídeo.
- **Distancia:** el 61 % de los vídeos consecutivos están a un paso. El resto queda detrás de una portada: subir K de 1 a 5 casi no ayuda, y K=20 solo alcanza el 87 % a cambio de muchos más pares de miniaturas.
- **Señal de miniaturas:** en `/yt` es del 100 % porque `recomendados()` las elige de la misma temática; en `/rtve` no supera al azar. En buena parte, estos pares reflejan cómo se genera la portada, no lo que quiere el usuario.
- **Clics:** una miniatura casi nunca precede a su vídeo, porque el simulador elige el vídeo sin mirar la portada.
- **Fallos:** ninguno es una miniatura. Sin miniaturas, un fallo puede relacionarse directamente con el vídeo anterior o siguiente de esa misma cara.

**Propuesta:** excluir las miniaturas de la señal de contexto; siguen disponibles en `seq`.

## 8. Pendiente

- **Miniaturas**: decidir si se confirma la propuesta de la sección 7 antes de fijar K.
- **Nombres**: profundidad, cardinalidad por nivel y componentes ruidosos (identificadores de vídeo de YouTube como `XuBC0NlmCpy`; no hay `seq=` ni `seg=`). La estructura depende del contenido: `/yt/canal/video`, `/vod/peli/titulo`, `/vod/serie/titulo/t1/e1`, y la miniatura de una serie cuelga de la temporada. Clave para relacionar nombres erróneos con los correctos, que nunca co-ocurren.
- **Popularidad**: distribución de peticiones por nombre (cola larga, contenidos virales como `/yt/shorts/RIgP_58waM-`).
- **Secuencias**: longitud por consumidor, para fijar K.
- **Repetición de pares**: dentro de una cara frente a entre caras, para fijar la saturación y el mínimo de caras distintas.
- **Tiempo**: forma del ajuste suave a partir de `hueco`.
- **Evaluación**: la iteración 2 usa la misma semilla, así que los nombres erróneos ya estarían en el entrenamiento. Decidirlo antes de citar resultados.